In [ ]:
import sys
sys.path.insert(0, "src")

import pandas as pd

from scrape_listing_details import enrich_listings

In [ ]:
listings = pd.read_json("output/listings.json", orient="records")

In [ ]:
listings

In [ ]:
# select listings with price <= 4000000, area_m2 >= 80, bathrooms >= 2, 
# parking_spots >= 1
listings = listings[
    (listings["price"] <= 4000000)
    & (listings["area_m2"] >= 75)
    & (listings["bathrooms"] >= 2)
    & (listings["parking_spots"] >= 1)
]

In [ ]:
urls = listings["url"].tolist()

In [ ]:
enriched_listings = enrich_listings(urls, headless=True)

In [ ]:
enriched_listings_df = pd.DataFrame(enriched_listings)
enriched_listings_df.to_json("output/enriched_listings.json", orient="records")

In [ ]:
# enriched_listings = enriched_listings[enriched_listings["estrato"]>=4]
enriched_listings_df["url"] = enriched_listings_df["url"].str.split("?").str[0]

display(enriched_listings_df)

# Plots

In [8]:
import pandas as pd


data = pd.read_json("output/merged_listings.json", orient="records")
data["url"] = data["url"].str.split("?").str[0]
data = data[data["estrato"] <= 6]

In [11]:
data.official_barrio.value_counts()

official_barrio
USAQUEN      24
SANTA ANA    20
Name: count, dtype: int64

In [10]:
data = data[
    data["official_barrio"].isin(
        [
            # "LOS CEDROS",
            # "LAS MARGARITAS",
            # "LA CASTELLANA",
            # "SALITRE OCCIDENTAL",
            # "MAZUREN",
            # "LOS ANDES",
            # "CEDRO SALAZAR",
            # "ACACIAS USAQUEN",
            # "CEDRITOS",
            # "BOSQUE CALDERON",
            # "NICOLAS DE FEDERMAN",
            # "PAULO VI",
            # "CAOBOS SALAZAR",
            "USAQUEN",
            "SANTA ANA",
            # "BATAN",
        ]
    )
]

In [12]:
import plotly.express as px
import plotly.graph_objects as go
from ipywidgets import HTML, VBox

# Sequential blue ramp (light -> dark), from the dataviz palette reference
blue_scale = [
    [0.00, "#cde2fb"],
    [0.14, "#b7d3f6"],
    [0.29, "#9ec5f4"],
    [0.43, "#86b6ef"],
    [0.57, "#6da7ec"],
    [0.71, "#5598e7"],
    [0.86, "#3987e5"],
    [1.00, "#0d366b"],
]

fig = px.scatter(
    data,
    x="area_m2",
    y="total_price",
    color="price_per_m2",
    color_continuous_scale=blue_scale,
    hover_data=["url", "official_barrio", "barrio", "estrato", "bathrooms", "parking_spots"],
    labels={
        "area_m2": "Area (m²)",
        "total_price": "Total price (COP)",
        "price_per_m2": "Price per m² (COP)",
    },
    title="Total price vs. area, colored by price per m²",
)
fig.update_traces(
    marker=dict(size=9, line=dict(width=1, color="rgba(255,255,255,0.6)"))
)
fig.update_layout(
    plot_bgcolor="#fcfcfb",
    paper_bgcolor="#fcfcfb",
    font_color="#0b0b0b",
    autosize=True,
    height=850,
    margin=dict(l=70, r=40, t=70, b=60),
)

fig_widget = go.FigureWidget(fig)
# Set after construction (not as a constructor kwarg, which gets validated
# against the Figure schema and rejected) so plotly.js re-measures its
# container via ResizeObserver instead of freezing the width detected at
# mount time — without it the widget keeps whatever width the output area
# happened to be when first rendered (e.g. half-width from a split editor).
fig_widget._config = {"responsive": True}
fig_widget.layout.autosize = True
fig_widget.layout.height = 650

link_display = HTML(value="<i>Click a point to see its listing URL here.</i>")


def _on_point_click(trace, points, state):
    if not points.point_inds:
        return
    idx = points.point_inds[0]
    url = trace.customdata[idx][0]
    link_display.value = f'<a href="{url}" target="_blank">{url}</a>'


for trace in fig_widget.data:
    trace.on_click(_on_point_click)

display(VBox([fig_widget, link_display], layout=dict(width="100%")))


    'data': [{'customdata': array([['https://www.metrocuadrado.com/inmueble/arri…